In [3]:
import pandas as pd

In [21]:
import requests
from bs4 import BeautifulSoup

In [26]:
import nltk
import string
import re
import os

In [49]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

In [48]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [11]:
input_df = pd.read_excel("Input.xlsx")

In [13]:
print(input_df)

In [19]:
for index, row in input_df.iterrows():
    url_id = row['URL_ID']
    url = row['URL']

    print(url_id, url)


In [74]:
os.makedirs("extracted_articles", exist_ok=True)

for index, row in input_df.iterrows():
    url_id = row['URL_ID']
    url = row['URL']

    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title
        title = soup.find('h1').get_text(strip=True)

        # Extract article text
        article_text = ""
        for p in soup.find_all('p'):
            article_text += p.get_text(strip=True) + " "

        # Combine title + article text
        full_text = title + "\n\n" + article_text

        # Step 6: TEXT CLEANING

        # 6a: Lowercase
        full_text = full_text.lower()

        # 6b: Remove extra spaces and line breaks
        full_text = re.sub(r'\s+', ' ', full_text)

        # 6c: Remove punctuation
        full_text = full_text.translate(str.maketrans('', '', string.punctuation))

        # 6d: Tokenize words
        words = word_tokenize(full_text)

        # 6e: Remove stopwords
        stop_words = set(stopwords.words('english'))
        cleaned_words = [w for w in words if w not in stop_words]

        # 6f: Rejoin cleaned words into text
        cleaned_text = " ".join(cleaned_words)

        # Step 7: Save cleaned text to file
        with open(f"extracted_articles/{url_id}.txt", "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"✅ Saved and cleaned article {url_id}")


    except Exception as e:
        print(f"Error with URL_ID {url_id}: {e}")

In [78]:
with open("positive-words.txt", "r") as f:
    positive_words = set([line.strip() for line in f if line.strip() and not line.startswith(';')])

with open("negative-words.txt", "r") as f:
    negative_words = set([line.strip() for line in f if line.strip() and not line.startswith(';')])

In [80]:
def count_syllables(word):
    word = word.lower()
    vowels = "aeiou"
    count = 0
    prev_char_was_vowel = False
    for char in word:
        if char in vowels:
            if not prev_char_was_vowel:
                count += 1
                prev_char_was_vowel = True
        else:
            prev_char_was_vowel = False
    if word.endswith("e"):
        count -= 1
    return max(count, 1)

In [82]:
def is_complex_word(word):
    return count_syllables(word) > 2


In [84]:
def count_personal_pronouns(text):
    # Use regex to avoid partial matches
    pronouns = re.findall(r'\b(i|we|my|ours|us)\b', text, flags=re.I)
    return len(pronouns)


In [92]:
folder_path = "extracted_articles"
output_rows = []

for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt"):
        url_id = os.path.splitext(file_name)[0]
        with open(os.path.join(folder_path, file_name), "r", encoding="utf-8") as f:
            text = f.read()

        words = word_tokenize(text)
        sentences = sent_tokenize(text)
        word_count = len(words)
        sentence_count = len(sentences)
        char_count = sum(len(word) for word in words)

        # Positive & Negative Score
        positive_score = sum(1 for w in words if w in positive_words)
        negative_score = sum(1 for w in words if w in negative_words)

        # Polarity & Subjectivity
        polarity_score = (positive_score - negative_score) / ((positive_score + negative_score) + 0.000001)
        subjectivity_score = (positive_score + negative_score) / (word_count + 0.000001)

        # Avg Sentence Length
        avg_sentence_length = word_count / (sentence_count + 0.000001)

        # Complex Words
        complex_words = [w for w in words if is_complex_word(w)]
        complex_word_count = len(complex_words)
        percentage_complex_words = (complex_word_count / (word_count + 0.000001)) * 100

        # Fog Index
        fog_index = 0.4 * (avg_sentence_length + percentage_complex_words)

        # Avg Number of Words per Sentence
        avg_words_per_sentence = word_count / (sentence_count + 0.000001)

        # Syllables per Word
        syllable_count = sum(count_syllables(w) for w in words)
        syllable_per_word = syllable_count / (word_count + 0.000001)

        # Personal Pronouns
        personal_pronouns = count_personal_pronouns(text)

        # Avg Word Length
        avg_word_length = char_count / (word_count + 0.000001)

        output_rows.append({
            "URL_ID": url_id,
            "POSITIVE SCORE": positive_score,
            "NEGATIVE SCORE": negative_score,
            "POLARITY SCORE": polarity_score,
            "SUBJECTIVITY SCORE": subjectivity_score,
            "AVG SENTENCE LENGTH": avg_sentence_length,
            "PERCENTAGE OF COMPLEX WORDS": percentage_complex_words,
            "FOG INDEX": fog_index,
            "AVG NUMBER OF WORDS PER SENTENCE": avg_words_per_sentence,
            "COMPLEX WORD COUNT": complex_word_count,
            "WORD COUNT": word_count,
            "SYLLABLE PER WORD": syllable_per_word,
            "PERSONAL PRONOUNS": personal_pronouns,
            "AVG WORD LENGTH": avg_word_length
        })

In [94]:
output_df = pd.DataFrame(output_rows)
output_df = output_df.sort_values("URL_ID")
output_df.to_excel("Output.xlsx", index=False)
print("✅ Text analysis complete. Output saved to Output.xlsx")